In [4]:
import os
import pandas as pd
import yaml

base_dir = "/home/vahid/Projects/pythonProject/PoC/experiments/gnn_run/edge_level/EllipticPP"
experiment_records = []
for root, dirs, files in os.walk(base_dir):
    if "hparams.yaml" in files and "metrics.csv" in files:
        hparams_path = os.path.join(root, "hparams.yaml")
        metrics_path = os.path.join(root, "metrics.csv")
        experiment_name = os.path.basename(os.path.dirname(root))  # e.g., the timestamp directory
        with open(hparams_path, "r") as f:
            hparams = yaml.safe_load(f)

        metrics_df = pd.read_csv(metrics_path)

        last_metrics = metrics_df.iloc[-1].to_dict()
        record = {
            "experiment": experiment_name,
            **hparams,
            **last_metrics,
        }

        experiment_records.append(record)

df = pd.DataFrame(experiment_records)
df = df.dropna(subset=['test_avg_pr'])
df.drop(['train_au_roc', 'train_aucnpr', 'step',
       'train_avg_pr', 'train_loss', 'val_au_roc', 'val_aucnpr', 'val_avg_pr',
       'val_loss'],axis=1,inplace=True)
df

,experiment,dataset_name,drop_ratio,dropout,epochs,gnn_type,hidden_size,learning_rate,num_layers,use_graph_features,epoch,gpu_memory_allocated_MB,gpu_memory_reserved_MB,test_au_roc,test_aucnpr,test_avg_pr,test_loss,time_sec
0,2025-07-07_19:31:24,EllipticPP,0.5,0.0,1000,GCN,32,0.005,2,True,1000.0,551.389160,1612.709839,0.988637,0.556892,0.557514,0.020936,0.015712
1,2025-07-07_17:54:33,EllipticPP,0.5,0.2,1000,GCN,32,0.005,1,False,1000.0,452.717560,1419.771851,0.987759,0.573868,0.574466,0.010398,0.015431
2,2025-07-07_18:37:57,EllipticPP,0.5,0.2,1000,GCN,64,0.005,1,False,1000.0,533.139465,2124.415039,0.985836,0.667560,0.668027,0.009991,0.018873
3,2025-07-07_20:58:00,EllipticPP,0.5,0.0,1000,GCN,128,0.005,2,True,1000.0,794.908691,3711.958984,0.952800,0.135640,0.136853,0.040832,0.025758
4,2025-07-07_19:20:33,EllipticPP,0.5,0.2,1000,GCN,128,0.005,1,False,1000.0,696.114197,3519.020752,0.983612,0.409258,0.410087,0.015268,0.020995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,2025-07-08_00:48:22,EllipticPP,0.5,0.0,1000,GAT,64,0.005,2,False,1000.0,533.173218,2118.123535,0.989023,0.652262,0.652750,0.010411,0.017329
71,2025-07-07_23:30:37,EllipticPP,0.5,0.2,1000,GAT,128,0.005,1,True,1000.0,794.776550,3716.153320,0.987945,0.624650,0.625177,0.013518,0.022033
72,2025-07-08_00:58:26,EllipticPP,0.5,0.2,1000,GAT,64,0.005,2,True,1000.0,631.835632,2319.450195,0.950508,0.280621,0.281630,0.025042,0.021976
73,2025-07-07_23:19:50,EllipticPP,0.5,0.0,1000,GAT,128,0.005,1,False,1000.0,696.114197,3514.826904,0.981238,0.679360,0.679810,0.009613,0.018206


In [21]:
import pandas as pd

# Sample structure for demonstration (replace with actual df if provided)
# Assuming df is already loaded

# Example column names based on user's description
group_cols = ['dataset_name', 'drop_ratio', 'dropout', 'epochs',
              'gnn_type', 'hidden_size', 'learning_rate', 'num_layers']
performance_metrics = ['test_au_roc', 'test_aucnpr', 'test_avg_pr', 'test_loss']

# Placeholder: generate example DataFrame if needed
# df = pd.read_csv("your_file.csv")  # replace this line with actual loading if needed

# Group by all config columns + use_graph_features to get performance
grouped = df.groupby(group_cols + ['use_graph_features'])[performance_metrics].mean().reset_index()

# Pivot so that use_graph_features becomes columns (False and True)
pivoted = grouped.pivot(index=group_cols, columns='use_graph_features', values=performance_metrics)

# Flatten column MultiIndex
pivoted.columns = [f"{metric}_{'with_graph' if use_graph else 'no_graph'}" for metric, use_graph in pivoted.columns]
pivoted = pivoted.reset_index()
pivoted

,dataset_name,drop_ratio,dropout,epochs,gnn_type,hidden_size,learning_rate,num_layers,test_au_roc_no_graph,test_au_roc_with_graph,test_aucnpr_no_graph,test_aucnpr_with_graph,test_avg_pr_no_graph,test_avg_pr_with_graph,test_loss_no_graph,test_loss_with_graph
0,PaySim,0.9,0.0,1000,GAT,32,0.005,1,0.817341,0.837867,0.537302,0.577112,0.539542,0.579160,8.660681,8.847404
1,PaySim,0.9,0.0,1000,GAT,32,0.005,2,0.852521,0.887037,0.592218,0.655080,0.594191,0.656750,10.496770,9.414028
2,PaySim,0.9,0.0,1000,GAT,64,0.005,1,0.865713,0.828877,0.678643,0.559213,0.680199,0.561347,11.605976,8.863414
3,PaySim,0.9,0.0,1000,GAT,64,0.005,2,0.831452,0.979323,0.566677,0.838739,0.568774,0.839519,8.573293,2.248750
4,PaySim,0.9,0.0,1000,GAT,128,0.005,1,0.874316,0.875864,0.603969,0.593246,0.605887,0.595215,27.116901,10.070847
5,PaySim,0.9,0.0,1000,GAT,128,0.005,2,0.864974,0.893503,0.574189,0.595563,0.576250,0.597520,8.383702,6.461135
6,PaySim,0.9,0.0,1000,GCN,32,0.005,1,0.865595,0.881598,0.659847,0.608604,0.661493,0.610498,3.977915,9.959909
7,PaySim,0.9,0.0,1000,GCN,32,0.005,2,0.926727,0.893414,0.375618,0.659609,0.378640,0.661257,8.201434,7.177241
8,PaySim,0.9,0.0,1000,GCN,64,0.005,1,0.865677,0.846819,0.676008,0.591361,0.677577,0.593339,3.095336,8.199221
9,PaySim,0.9,0.0,1000,GCN,64,0.005,2,0.856717,0.824650,0.656266,0.590787,0.657930,0.592768,6.778059,7.307279
